# 08 - REGISTRY: Registro del modelo en Unity Catalog / Model registry in Unity Catalog

## SPA:
**Propósito**
Llevar el modelo elegido en 07_mlflow desde el experimento al registro de Unity Catalog, darle la etiqueta champion, documentarlo y comprobar que se carga y predice igual que el original.

**Entrada**
El run ganador del experimento telco-churn, identificado como el único run con métrica de test.

**Salida**
Modelo registrado `telco_churn.models.churn_classifier`, versión 1, con alias `champion`. Este notebook no escribe tablas.

**Decisiones de diseño**
- El run ganador se busca con un filtro y no con un identificador copiado a mano, porque los identificadores cambian cada vez que se reejecuta 07_mlflow.
- El registro está protegido contra duplicados: si ya existe una versión procedente del mismo run, se reutiliza. Sin esa comprobación, cada Run all crearía una versión nueva idéntica.
- Se usa el alias champion en lugar de los stages antiguos, que no existen en Unity Catalog. El código que carga el modelo pide el champion, así que promocionar una versión nueva consiste en mover la etiqueta sin tocar ese código.
- Las descripciones del modelo y de la versión se escriben desde el notebook, con las métricas leídas del propio run.

**Notas**
- El notebook instala al principio las versiones de requirements.txt. Sin ellas, el entorno serverless traía scikit-learn 1.7.2 y el modelo, entrenado con la 1.6.1, no se podía cargar.
- En el esquema models aparece también una función llamada churn_classifier que no crea ninguna celda. Probablemente es la forma en que Unity Catalog representa internamente el modelo registrado, aunque no está confirmado. No se elimina.

## ENG:
**Purpose**
Move the model chosen in 07_mlflow from the experiment into the Unity Catalog registry, give it the champion alias, document it and check that it loads and predicts exactly like the original.

**Input**
The winning run from the telco-churn experiment, identified as the only run with a test metric.

**Output**
Registered model `telco_churn.models.churn_classifier`, version 1, with alias `champion`. This notebook does not write any tables.

**Design decisions**
- The winning run is found with a filter rather than an ID copied by hand, because run IDs change every time 07_mlflow is re-run.
- Registration is protected against duplicates: if a version from the same run already exists, it is reused. Without that check, every Run all would create a new, identical version.
- The champion alias is used instead of the old stages, which do not exist in Unity Catalog. Code that loads the model asks for the champion, so promoting a new version means moving the alias without touching that code.
- The model and version descriptions are written from the notebook, with the metrics read from the run itself.

**Notes**
- The notebook installs the versions from requirements.txt at the start. Without them, the serverless environment shipped scikit-learn 1.7.2 and the model, trained with 1.6.1, could not be loaded.
- The models schema also shows a function called churn_classifier that no cell creates. It is probably how Unity Catalog represents the registered model internally, although this has not been confirmed. It is not deleted.

In [0]:
%pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
from mlflow import MlflowClient
from churn.config import MLFLOW_EXPERIMENT_NAME, MODEL_NAME

mlflow.set_registry_uri("databricks-uc")

usuario = spark.sql("SELECT current_user()").collect()[0][0]
experimento = mlflow.set_experiment(f"/Users/{usuario}/{MLFLOW_EXPERIMENT_NAME}")

client = MlflowClient()

In [0]:
ganadores = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id],
    filter_string="metrics.pr_auc_test > 0",
)

assert len(ganadores) == 1, f"Se esperaba un solo run con métrica de test, hay {len(ganadores)}"

run_id_ganador = ganadores.loc[0, "run_id"]

display(ganadores[["tags.mlflow.runName", "metrics.pr_auc_cv_media", "metrics.pr_auc_test"]])

In [0]:
model_uri = f"runs:/{run_id_ganador}/modelo"

versiones = client.search_model_versions(f"name='{MODEL_NAME}'")
ya_registrada = [v for v in versiones if v.run_id == run_id_ganador]

if ya_registrada:
    version = ya_registrada[0]
else:
    version = mlflow.register_model(model_uri, MODEL_NAME)

print(version.name, version.version)

In [0]:
client.set_registered_model_alias(MODEL_NAME, "champion", version.version)

champion = client.get_model_version_by_alias(MODEL_NAME, "champion")

print(champion.version, champion.run_id == run_id_ganador)

In [0]:
client.update_registered_model(
    name=MODEL_NAME,
    description=(
        "Predice la probabilidad de que un cliente de Telco se dé de baja en el próximo mes. "
        "Entrada: las 26 columnas de telco_churn.gold.customer_features, sin customer_id ni churn. "
        "Salida: probabilidad de quedarse y de irse. El umbral de decisión se aplica fuera del modelo; "
        "rango candidato de 0,35 a 0,45. Métrica de selección: PR-AUC."
    ),
)

fila = ganadores.loc[0]

client.update_model_version(
    name=MODEL_NAME,
    version=version.version,
    description=(
        f"Regresión logística sin ajuste de hiperparámetros. "
        f"PR-AUC en validación cruzada: {fila['metrics.pr_auc_cv_media']:.4f}. "
        f"PR-AUC en test: {fila['metrics.pr_auc_test']:.4f}. "
        f"Run de origen: {run_id_ganador}."
    ),
)

In [0]:
import numpy as np
from churn.config import GOLD_TABLE, ID, TARGET

pdf = spark.table(GOLD_TABLE).toPandas()
muestra = pdf.drop(columns=[ID, TARGET]).head(200)

modelo_champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
modelo_run = mlflow.pyfunc.load_model(model_uri)

proba_champion = np.asarray(modelo_champion.predict(muestra))
proba_run = np.asarray(modelo_run.predict(muestra))

print(proba_champion[:3])
assert np.allclose(proba_champion, proba_run), "El champion no reproduce el modelo del run"